# End-to-End Competition Workflow — From Notebook to Kaggle Submission

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb18_competition_workflow_student.ipynb)


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Complete both to receive participation credit. The second exercise produces the `submission.csv` you upload to the Kaggle leaderboard.


This notebook presents an **end-to-end predictive modeling workflow**, using our course competition as a practical example. It threads together everything from nb01–nb17: the train/test split discipline, preprocessing pipelines, the logistic baseline, the tree ensembles, cross-validation with Student's *t* confidence intervals, and the model-selection protocol.

Because the competition requires reproducibility, we fix `RANDOM_SEED = 474` (the course seed). The competition also provides a `test.csv` file without outcome labels. For that reason, `test.csv` is only used at the very end — to generate our final prediction file (`submission.csv`) after we retrain our selected model on the entire training set.

The modeling process follows this structure:

* `train.csv` is split once into an 80% **training set** (for 5-fold cross-validation and model selection) and a 20% **held-out test set** (for one unbiased check of the champion).
* Once the champion is selected and checked on the held-out set, the two splits are **merged** and the model is **retrained on the entire `train.csv`** before generating predictions for `test.csv`.

Our workflow is organized in the following sequence. For each step we also provide a **Gemini Prompt** suggestion you can copy into Gemini to request the corresponding Python code.

- Step 0 — Setup, configuration, and imports
- Step 1 — Data loading and sanity checks
- Step 2 — Exploratory Data Analysis (EDA)
- Step 3 — Data preparation and feature engineering
- Step 4 — Baseline model (Logistic Regression)
- Step 5 — More complex models (LASSO, Random Forest, Gradient Boosting)
- Step 6 — Compare models with visualization
- Step 7 — Model selection and rationale
- Step 8 — Evaluate the champion on the held-out test set
- Step 9 — Final training on the full (merged) training data
- Step 10 — Predict on the competition test set and export `submission.csv`
- Step 99 — Reporting and reproducibility

## Step 0 — Setup, configuration, and imports

> 💡 **Gemini Prompt:** "Write a setup cell for a binary-classification competition workflow. Import pandas, numpy, matplotlib.pyplot, joblib, `scipy.stats`, and from scikit-learn: `ColumnTransformer`, `Pipeline`, `StandardScaler`, `OneHotEncoder`, `SimpleImputer`, `StratifiedKFold`, `cross_val_score`, `cross_val_predict`, `GridSearchCV`, `LogisticRegression`, `RandomForestClassifier`, `GradientBoostingClassifier`, `roc_auc_score`, and `classification_report`. Set `RANDOM_SEED = 474`, seed numpy, set the figure size to (10, 6) and display precision to 3, then define a helper `ci95(mean, sd, k=5)` returning the Student's-t 95% CI half-width and bounds."
>
> **After running, verify:**
> - The cell prints `RANDOM_SEED = 474` with no import errors.
> - `ci95(0.80, 0.02)` returns a half-width near 0.025 (the t-multiplier for df=4 is \~2.776).

In [ ]:
# YOUR CODE HERE

## Step 1 — Data loading and sanity checks

First, **upload the competition data to Colab.** Open the **Files** panel (the folder icon in the left sidebar), click the upload button, and add `train.csv` and `test.csv` from the case-competition starter pack. Uploaded files land in Colab's working directory (`/content`), so we can load them by name.

We load the labeled `train.csv` and the unlabeled `test.csv`, then run the sanity checks that keep the rest of the notebook honest: shapes, dtypes, missing-value counts, and the target class balance. Sections 2–8 work strictly on the labeled training data; `test.csv` is opened only in Step 9 to generate the submission.

> 💡 **Gemini Prompt:** "After uploading `train.csv` and `test.csv` to the Colab Files panel, load them from the current working directory into `df_train_full` and `df_test_kaggle`. The target column is `Exited`. Raise a clear `FileNotFoundError` (naming `os.getcwd()`) if a file is missing. Print both shapes, the dtypes, missing-value counts, and the target class balance for the training set, and confirm the only column difference between train and test is the target."
>
> **After running, verify:**
> - Both files load without dtype warnings (or the warning is named in your README).
> - The target balance is around 0.20 — the dataset is moderately imbalanced.
> - `set(train.columns) - set(test.columns)` is exactly `{'Exited'}`.

In [ ]:
# YOUR CODE HERE

## Step 2 — Exploratory Data Analysis (EDA)

A compact three-panel EDA snapshot — class balance, missingness, and the numeric features most correlated with the target — is enough to confirm the dataset matches the rubric's description and to flag any column to drop or transform. We use matplotlib throughout, matching the course's plotting convention from nb01 onward.

> 💡 **Gemini Prompt:** "From `df_train_full`, make one matplotlib figure with three subplots: (a) `Exited` value counts as a bar chart; (b) per-column missing-value counts, sorted descending, top 10; (c) absolute correlation of each numeric column with `Exited`, top 10. Use `plt.tight_layout()`."
>
> **After running, verify:**
> - The class-balance panel shows the \~20% minority class.
> - The correlation panel ranks the usual churn drivers (e.g., `Age`, `Balance`, `NumOfProducts`) near the top.

In [ ]:
# YOUR CODE HERE

## Step 3 — Data preparation and feature engineering

We wrap every column transformation inside a `ColumnTransformer` (nb02), so the exact logic that fits on training data is the logic that predicts on the Kaggle test set. Numeric columns are median-imputed and standardized; categorical columns are mode-imputed and one-hot encoded with `handle_unknown='ignore'`. We also drop identifier columns so they never leak into the model.

We then make **one stratified split** of `train.csv` (nb01/nb03): 80% becomes the **training set** used for cross-validation and model selection, and 20% becomes a **held-out test set** we touch only once, at the end, to get an unbiased estimate of the champion's performance. After selection we merge the two back together and refit on all of `train.csv` before predicting the competition `test.csv`.

> 💡 **Gemini Prompt:** "Write `build_preprocessor(df, target='Exited')` that drops identifier-like columns (`RowNumber`, `CustomerId`, `Surname`, `id`) if present, then returns a `ColumnTransformer`: numeric columns get median `SimpleImputer` + `StandardScaler`; categorical columns get most-frequent `SimpleImputer` + `OneHotEncoder(handle_unknown='ignore')`. Build the full feature matrix `X_full, y_full`, then make one stratified 80/20 `train_test_split` into `X_train, X_holdout, y_train, y_holdout` with `random_state=474`."
>
> **After running, verify:**
> - The categorical list includes `Geography` and `Gender`; identifiers are excluded.
> - `X_train` has \~80% of the rows and `X_holdout` \~20%, both with the same class balance (stratified).

In [ ]:
# YOUR CODE HERE

## Step 4 — Baseline model (Logistic Regression)

We anchor the comparison with a **logistic-regression baseline** (nb06), evaluated with 5-fold stratified cross-validation on the training data only (nb08). We report the mean ROC-AUC with a **Student's *t* 95% confidence interval** — `mean ± t_crit · sd / √k` — exactly the interval the course has used since nb08. Out-of-fold predictions give an honest `classification_report` in which every prediction comes from a fold that never saw that row during fitting.

> 💡 **Gemini Prompt:** "Build a pipeline of `build_preprocessor` + `LogisticRegression(max_iter=2000, class_weight='balanced')`. Score it with `cross_val_score(..., scoring='roc_auc', cv=cv_strat)`, report the mean ROC-AUC and its Student's-t 95% CI via `ci95`, then use `cross_val_predict` to print a `classification_report`."
>
> **After running, verify:**
> - Baseline CV ROC-AUC lands near 0.75–0.82.
> - The classification report's recall on the churn class reflects the `class_weight='balanced'` setting.

In [ ]:
# YOUR CODE HERE

## Step 5 — More complex models (LASSO, Random Forest, Gradient Boosting)

Building on the logistic baseline, we challenge it with three models the course has already taught: a **LASSO (L1-penalized) logistic regression** (nb05), a **Random Forest** (nb12), and a **Gradient Boosting** classifier (nb13). Each is tuned with a small `GridSearchCV` over the hyperparameters that mattered in those notebooks (nb09), and each is scored on the same 5-fold stratified CV so the ROC-AUC numbers are directly comparable. We keep every model inside the same preprocessing pipeline to prevent leakage.

### Step 5.1 — LASSO (L1-penalized logistic regression)

> 💡 **Gemini Prompt:** "Build a pipeline of `build_preprocessor` + `LogisticRegression(penalty='l1', solver='liblinear', class_weight='balanced')`. Tune `C` over `[0.01, 0.1, 1, 10]` with `GridSearchCV(scoring='roc_auc', cv=cv_strat)`. Report the best `C`, the best mean ROC-AUC, and its Student's-t 95% CI using the fold standard deviation from `cv_results_`."
>
> **After running, verify:**
> - A best `C` is printed and the LASSO CV ROC-AUC is in the same ballpark as the logistic baseline.
> - Small `C` (stronger L1) zeros out some coefficients — the regularization lesson from nb05.

In [ ]:
# YOUR CODE HERE

### Step 5.2 — Random Forest

> 💡 **Gemini Prompt:** "Build a pipeline of `build_preprocessor` + `RandomForestClassifier(class_weight='balanced', n_jobs=-1)`. Tune `n_estimators` over `[200]`, `max_depth` over `[3, 5, None]`, and `max_features` over `['sqrt', 0.5]` with `GridSearchCV(scoring='roc_auc', cv=cv_strat)`. Report the best parameters, the best mean ROC-AUC, and its Student's-t 95% CI."
>
> **After running, verify:**
> - The forest's CV ROC-AUC typically edges above the logistic baseline.
> - The best `max_features` is `'sqrt'` or `0.5` — the de-correlation knob from nb12.

In [ ]:
# YOUR CODE HERE

### Step 5.3 — Gradient Boosting

> 💡 **Gemini Prompt:** "Build a pipeline of `build_preprocessor` + `GradientBoostingClassifier()`. Tune `n_estimators` over `[200]`, `learning_rate` over `[0.05, 0.2]`, and `max_depth` over `[3]` with `GridSearchCV(scoring='roc_auc', cv=cv_strat)`. Report the best parameters, the best mean ROC-AUC, and its Student's-t 95% CI."
>
> **After running, verify:**
> - The boosted model is usually the top or near-top mean ROC-AUC.
> - A smaller `learning_rate` with enough trees is the disciplined setting from nb13.

In [ ]:
# YOUR CODE HERE

## Step 6 — Compare models with visualization

We assemble the four candidates into one comparison table (mean ROC-AUC, fold SD, and Student's-t 95% CI half-width) and plot them with error bars. Reading the chart is the whole point: when two models' CIs overlap, their performance is statistically indistinguishable and parsimony should decide.

> 💡 **Gemini Prompt:** "Build a DataFrame `results` with one row per model (`Logistic`, `LASSO`, `Random Forest`, `Gradient Boosting`) holding `roc_auc_mean`, `roc_auc_sd`, and the Student's-t 95% half-width from `ci95`. Sort by mean descending and draw a matplotlib bar chart of the means with the half-width as symmetric error bars."
>
> **After running, verify:**
> - Four bars appear, each with an error bar.
> - The table is sorted with the highest-mean model first.

In [ ]:
# YOUR CODE HERE

## Step 7 — Model selection and rationale

We pick the champion with the same **CI-overlap → parsimony** rule the model-selection protocol introduced in nb14. The top model by mean wins outright only if its 95% CI lower bound clears the runner-up's lower bound. If the CIs overlap, the two are statistically tied, so the **simpler** model wins — interpretability and deployment cost break the tie. This is the discipline that stops us from chasing a fractionally higher leaderboard score with a heavier model we cannot defend.

> 💡 **Gemini Prompt:** "Given the sorted `results` table, apply the CI-overlap rule: compute each model's 95% CI lower bound; if the top model's lower bound exceeds the runner-up's, select the top model; otherwise select the simpler of the two using a complexity ranking (Logistic < LASSO < Random Forest < Gradient Boosting). Print the champion and a one-paragraph rationale, and set `model_choice`."
>
> **After running, verify:**
> - A single `champion_name` and `model_choice` are printed.
> - The printed rationale references the CI comparison, not just the raw means.

In [ ]:
# YOUR CODE HERE

## 📝 PAUSE-AND-DO Exercise 1 — Pick Your Champion (10 minutes)

**Task:** Using the comparison table and Student's-t confidence intervals from Steps 6–7, decide which of the four candidates — **Logistic Regression**, **LASSO**, **Random Forest**, or **Gradient Boosting** — is your competition champion. Justify your choice in three sentences, then confirm `model_choice` matches your decision before running Step 8.

### YOUR CHAMPION DECISION HERE:

**Champion model:** *[logistic / lasso / rf / gbm]*

**Justification (3 sentences):**

1. *[CV ROC-AUC comparison — which candidate has the highest mean, and by how much?]*
2. *[CI overlap or separation — does the top model's 95% CI clear the runner-up's, or do they overlap so the simpler model wins by parsimony?]*
3. *[Business rationale — interpretability, deployment cost, calibration concerns.]*

## Step 8 — Evaluate the champion on the held-out test set

Before shipping, we get one **unbiased** estimate of the champion's competition performance. We fit the selected pipeline on the 80% training split (`X_train`) and score it **once** on the 20% held-out test split (`X_holdout`) that no cross-validation fold ever saw. This is the same discipline as nb14's test-set ceremony: the held-out set is opened a single time, for the chosen model only.

We then reuse **nb14's ceremony plot** — the champion's cross-validated mean with its 95% confidence interval drawn as a grey error bar, and the held-out test point laid over it as a triangle — and read the **INSIDE / ABOVE / BELOW** verdict. A test point INSIDE the CV interval means the estimate held and the model is ready to ship; BELOW warns the model may have overfit the training data; ABOVE is a pleasant surprise worth understanding before you trust it.

> 💡 **Gemini Prompt:** "Write `build_champion(model_choice)` returning the pipeline (`build_preprocessor` + the chosen classifier with its tuned hyperparameters). Fit it on `X_train, y_train` and compute the held-out ROC-AUC `roc_auc_score(y_holdout, pipe.predict_proba(X_holdout)[:, 1])`. Pull the champion's CV mean and SD from the Step 6 `results` table, build its 95% CI with `ci95`, and decide an INSIDE / ABOVE / BELOW verdict. Then recreate nb14's ceremony plot: a grey error-bar marker at the CV mean with its 95% CI, a triangle at the held-out test point colored by the verdict (green INSIDE, blue ABOVE, red BELOW), and dotted CI bounds."
>
> **After running, verify:**
> - The held-out ROC-AUC and an INSIDE / ABOVE / BELOW verdict are printed for the champion only.
> - The plot shows the CV mean ± 95% CI with the held-out test point overlaid.

In [ ]:
# YOUR CODE HERE

## Step 9 — Final training on the full (merged) training data

With the champion chosen and its held-out performance confirmed, we **merge the training and held-out test splits back into the full `train.csv`** and refit the champion on all of it — squeezing out every labeled row before predicting the competition test set. We rebuild the winning pipeline from `model_choice`, fit it on `X_full, y_full`, then persist it with `joblib` (nb02) so a teammate — or a grader — can reload and reproduce predictions without retraining.

> 💡 **Gemini Prompt:** "Reuse `build_champion(model_choice)` from Step 8 to rebuild the champion pipeline, refit it on the merged full training data `X_full, y_full`, then `joblib.dump` it to `artifacts/champion_pipeline.joblib` and print the file size."
>
> **After running, verify:**
> - The artifact file is written and its size prints in KB/MB.
> - The pipeline matches `model_choice` from Step 7 and is now fit on every labeled row.

In [ ]:
# YOUR CODE HERE

## Step 10 — Predict on the competition test set and export `submission.csv`

We reload the saved pipeline (the way a grader would) and run it on the **unlabeled** `test.csv`. This is the one place the test file is touched — and it is a **production prediction, not a model evaluation**: the file has no labels, so we cannot and do not score it. The `submission.csv` must have exactly the two columns the leaderboard expects: the row identifier and the predicted probability of `Exited`.

> 💡 **Gemini Prompt:** "Reload the pipeline from `ARTIFACT_PATH`. Detect the id column in `df_test_kaggle` (one of `id`, `CustomerId`, `RowNumber`). Call `predict_proba` on the test features and build a DataFrame with columns `[id, Exited]` where `Exited` is the class-1 probability. Save it as `submission.csv` with `index=False` and print the head."
>
> **After running, verify:**
> - `submission.csv` row count equals `df_test_kaggle` row count.
> - The `Exited` column is a float in [0, 1], not 0/1.

In [ ]:
# YOUR CODE HERE

## 📝 PAUSE-AND-DO Exercise 2 — File the Submission (10 minutes)

**Task:** Confirm the `submission.csv` produced in Step 9 is leaderboard-ready, then upload it.

**Checklist:**

- [ ] `submission.csv` has the exact column names the competition page lists (case-sensitive).
- [ ] Row count matches the `test.csv` row count.
- [ ] The predicted-probability column is a float in `[0, 1]`, not 0/1.
- [ ] No `NaN` rows — `submission.isna().sum().sum() == 0`.
- [ ] The first three rows look like `(id, probability)` pairs you would expect.

Once those five checks pass, upload `submission.csv` to the Kaggle leaderboard.

## Step 99 — Reporting and reproducibility

**Executive summary (template):** dropped identifier columns (`RowNumber`, `CustomerId`, `Surname`); compared a logistic baseline against a tuned LASSO, Random Forest, and Gradient Boosting; selected the champion with the CI-overlap → parsimony rule from nb14.

**Assumptions:** median/most-frequent imputation; one-hot encoding with `handle_unknown='ignore'`; `class_weight='balanced'`; 5-fold stratified CV; Student's-t 95% confidence intervals (`stats.t.ppf(0.975, df=4)`); `RANDOM_SEED = 474`.

**How to run:**
- Place `train.csv` and `test.csv` at `DATA_DIR`.
- Run all cells top to bottom.
- Artifacts: `artifacts/champion_pipeline.joblib`, `submission.csv`, `requirements.txt`.

> 💡 **Gemini Prompt:** "Write a `requirements.txt` capturing the installed versions of pandas, numpy, scikit-learn, scipy, matplotlib, and joblib so the environment is reproducible. Read each version at runtime (e.g., `pd.__version__`) and print the file contents."

In [ ]:
# YOUR CODE HERE

## Wrap-Up — Key Takeaways

1. **One workflow, nine steps.** Setup → load → EDA → feature engineering → baseline → richer models → comparison → selection → final fit → submission. Every case-competition deliverable lives somewhere on this spine; if you can name the step, you can debug the failure.
2. **Compare models with confidence intervals, not point estimates.** The 5-fold CV ROC-AUC and its Student's-t 95% CI are what justify promoting LASSO, a Random Forest, or Gradient Boosting over the logistic baseline. When the CIs overlap, the simpler model wins by parsimony.
3. **Selection is a documented decision, not a leaderboard reflex.** Step 7 applies nb14's CI-overlap rule and writes the rationale in plain English. A model you can defend in three sentences beats a slightly higher public-leaderboard score you cannot explain.
4. **Split, select, check, then merge.** One 80/20 split lets cross-validation pick the champion on the training portion and gives an unbiased held-out check before you commit. Only then do you merge both splits, refit on the entire `train.csv`, and touch the unlabeled `test.csv` — a production prediction, not a model evaluation.
5. **The submission file is a contract.** Exact column names, one probability per test row, no `NaN`. Persist the fitted pipeline with `joblib` so a teammate — or a grader — can reproduce your `submission.csv` in seconds.

> **A question that often comes up here:** *"What if my Kaggle score is much worse than my cross-validated AUC?"* Three usual culprits, in order of frequency: (1) the train and test files have different column distributions ("covariate shift") — compare numeric means and category counts; (2) preprocessing transformed a column on training data that behaves differently on test data — check that the pipeline's transformed test matrix has the same shape as the training transform; or (3) a leaky feature inflated your CV folds — if a single feature looks too good to be true, it probably is.


## Participation Assignment Submission Instructions

1. **Complete both PAUSE-AND-DO exercises**.
2. **Run all cells**.
3. **Save with output** and submit `nb18_competition_workflow_<your_lastname>.ipynb` to Brightspace.
4. **Upload `submission.csv`** to the Kaggle leaderboard.

### Next Step:

- **Notebook 19** — Deep learning horizons (Day 19)

**Bibliography**
- scikit-learn User Guide — [Model persistence](https://scikit-learn.org/stable/model_persistence.html)


<center>

# Thank you!

</center>
